In [20]:
import pandas as pd
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from pathlib import Path
from sentence_transformers import SentenceTransformer
from typing import Literal
from ollama import chat
import pymupdf
import os
import re
import numpy as np
import json

In [21]:
FOLDER_PATH = "/Users/tomthomas/Documents/rag-foundation/data/papers"
paths = list(Path(FOLDER_PATH).glob("*.pdf"))
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True, device="cpu")
OUTPUT_PATH = "/Users/tomthomas/Documents/rag-foundation/output"

<All keys matched successfully>


In [22]:
class Chunk(BaseModel):
    text: str
    chunk_id: str
    section_tag: Literal["References", "Content", "Appendix"]
    page_number: int = Field(ge=1)
    doc_id: str

class RetrievedChunk(BaseModel):
    chunk: Chunk
    score: float

class Citation(BaseModel):
    page_number: int = Field(ge=1)
    doc_id: str

class AnswerSentence(BaseModel):
    text: str
    citations: list[Citation]

class Answer(BaseModel):
    sentences: list[AnswerSentence] = Field(min_length=1)

In [23]:
def is_appendix_heading(lines: list[str], i: int) -> bool:
    s = lines[i].strip()
    if s.lower() == "appendix":
        return True
    if s == "A":
        for nxt in lines[i + 1:]:
            nxt = nxt.strip()
            if nxt:                      # first non-empty line after the "A"
                return nxt[0].isupper()
    return False

In [24]:
def load_pages(path: str) -> list[str]:
    """Returns list of pages from the given file path.
    
    Args:
        path (str): The file path to each paper.
    
    The page number of item i is i + 1 (pages are numbered from 1, matching the PDF page index).
    Returns:
        list: A list of pages from the given file path.
    
    """
    doc = pymupdf.open(path)
    pages = []
    for page in doc:
        pages.append(page.get_text())
    doc.close()
    return pages


In [25]:
def split_sections(pages: list[str]) -> dict[str, list[tuple[int, str]]]:
    """Returns a dictionary containing section and section pages.
        
        Args:
            pages (list[str]): list of pages from the document.

        Definition:
            A section is defined as a part of the document that starts with a section heading and ends before the next section heading.
            If no heading, then the section is by default content for the text.
            If reference is in middle of page, text after the reference is considered part of the reference section.
        
        Returns:
            dict: A dictionary containing section and section pages, where the key is the section name and the value is a list of tuples containing the page number and the text on that page.
        
        """
    sections = {
        "Content": [],
        "References": [],
        "Appendix": []
    }
    ref_found = False
    appendix_found = False
    for page_num, page in enumerate(pages, start=1):
        ref_text = ""
        appendix_text = ""
        content = ""
        lines = page.split("\n")
        for i, line in enumerate(lines):
            if ref_found and not is_appendix_heading(lines, i):
                ref_text += line + "\n"
            elif appendix_found and line.strip().lower() != "references":
                appendix_text += line + "\n"
            elif line.strip().lower() == "references":
                if appendix_found:
                    appendix_found = False
                ref_found = True
                ref_text += line + "\n"
            elif is_appendix_heading(lines, i) and ref_found:
                if ref_found:
                    ref_found = False
                appendix_found = True
                appendix_text += line + "\n"
            else:
                content += line + "\n"
        if content != "":
            sections["Content"].append((page_num, content))
        if ref_text != "":
            sections["References"].append((page_num, ref_text))
        if appendix_text != "":
            sections["Appendix"].append((page_num, appendix_text))
    return sections

In [26]:
def clean_section(section: list[tuple[int, str]]) -> list[tuple[int, list[str]]]:
    """Cleans a section by removing any empty text entries/removing unwanted new line strings between sentences 
        or removing emtpy lines or removing unwanted data (for eg: lines with only digits or removing roman letters etc.).

        Args:
            section (list[tuple[int, str]]): A list of tuples containing the page number and the text on that page.

        Returns:
            list[tuple[int, list[str]]]: A cleaned list of tuples containing the page number and a list of paragraphs on that page.

    """

    cleaned_section = []
    roman_numeral = re.compile(
        r"(?i)M{0,3}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})"
    )
    for page_number, text in section:
        sentences = []
        page_text = ""
        text = text.split("arXiv Template\nA PREPRINT\n")[-1]
        lines = text.split("\n")
        non_empty = [i for i, l in enumerate(lines) if l.strip()]
        if non_empty:
            first, last = non_empty[0], non_empty[-1]
        else:
            first, last = 0, len(lines) - 1
        for i, line in enumerate(lines):
            line = line.strip()
            if i == first and (line.isdigit() or line.startswith("Published as a conference paper at")):
                continue
            if i == last and line.isdigit():
                continue
            page_text += line + "\n"
        paragraph = re.split(r"(?<=\.)\n", page_text)
        for sentence in paragraph:
            sentence = sentence.strip()
            sentence = sentence.replace("-\n", "")
            idx = sentence.find("\n")
            if idx != -1:
                sentence = sentence.replace("\n", " ")
            if (sentence.isdigit() or sentence.isspace() or not sentence.strip() or sentence.startswith(("arXiv","Preprint","URL","org","doi")) or ("arXiv" in sentence) or (roman_numeral.fullmatch(sentence)) or (sentence.isalpha() and len(sentence) ==1)) and (len(sentence) < 50):
                continue
            sentences.append(sentence)
        if sentences:
            cleaned_section.append((page_number, sentences))
    return cleaned_section


In [27]:
def chunk_paragraphs(section: list[tuple[int, list[str]]], section_tag: str, doc_id: str, chunk_size: int = 600, chunk_overlap: int = 150) -> list[Chunk]:
    """Chunks a section paragraphs.

        Args:
            section (list[tuple[int, list[str]]]): A list of tuples containing the page number and a list of paragraphs on that page.
            section_tag (str): The tag or name of the section. Allowed values: ["References", "Content", "Appendix"]
            doc_id (str): The ID of the document.
            chunk_size (int, optional): The maximum size of each chunk in characters. Defaults to 600.
            chunk_overlap (int, optional): The number of characters to overlap between chunks. Defaults to 150.

        Paragraph upto chunk_size character is considered as a single chunk.
        If a paragraph short of chunk_size characters, it will be combined with the next paragraph to form a chunk of up to chunk_size characters, if the second paragraph can be added without breaking sentences.
        If a paragraph exceeds chunk_size characters, it will be split into multiple chunks, ensuring that each chunk ends at a sentence boundary and the succeding chunk starts with an overlap of chunk_overlap characters from preceeding chunk without breaking a sentence with a 10% character tolerance.
        if a chunk spans multiple pages, the page number will be recorded as the starting page.
        The chunking logic ensures that sentences are not broken and that overlaps are maintained for context continuity.
        Returns:
            list[Chunk]: A list of Chunk objects containing the chunked paragraphs, chunk id, section_tag, page number and doc_id.

    """
    flat = []
    for page_number, paragraphs in section:
        for paragraph in paragraphs:
            sentences = re.split(r"(?<!\b[A-Z]\.)(?<=[.!?])\s+(?=[A-Z])", paragraph)
            for sentence in sentences:
                if len(sentence) <= chunk_size:
                    flat.append((page_number, sentence))
                else:
                    words = sentence.split(" ")
                    current_chunk = ""
                    for word in words:
                        if len(current_chunk) + len(word) + 1 <= chunk_size:
                            current_chunk += word + " "
                        else:
                            if current_chunk:
                                flat.append((page_number, current_chunk.strip()))
                            current_chunk = word + " "
                    if current_chunk:
                        flat.append((page_number, current_chunk.strip()))
    buffer = []
    counter = 0
    chunks = []
    for page_number, sentence in flat:
        string = " ".join([s for _, s in buffer])
        if not buffer:
            size = len(string) + len(sentence)
        else:
            size = len(string) + len(sentence) + 1
        if size <= chunk_size:
            buffer.append((page_number, sentence))
        elif size > chunk_size and buffer:
            chunk_text = " ".join([s for _, s in buffer])
            chunk = Chunk(
                text=chunk_text,
                chunk_id=doc_id+f"_{section_tag}_{counter:04d}",
                section_tag=section_tag,
                page_number=buffer[0][0],
                doc_id=doc_id
            )
            pg_no, txt = buffer[-1]
            buffer = []
            if len(txt) <= chunk_overlap * 1.10 and len(sentence + txt) + 1 <= chunk_size:
                buffer.append((pg_no, txt))
            counter += 1
            buffer.append((page_number, sentence))
            chunks.append(chunk)
    if buffer:
        chunk = Chunk(
            text=" ".join([s for _, s in buffer]),
            chunk_id=doc_id+f"_{section_tag}_{counter:04d}",
            section_tag=section_tag,
            page_number=buffer[0][0],
            doc_id=doc_id
        )
        chunks.append(chunk)
    return chunks

        

In [28]:
def semantic_chunking(section: list[tuple[int, list[str]]], section_tag: str, doc_id: str, chunk_size: int = 600, chunk_overlap: int = 150, min_chunk_size: int = 200, similarity_threshold: float = 0.5, debug: bool = False) -> list[Chunk]:
    """Chunks a section paragraphs.

        Args:
            section (list[tuple[int, list[str]]]): A list of tuples containing the page number and a list of paragraphs on that page.
            section_tag (str): The tag or name of the section. Allowed values: ["References", "Content", "Appendix"]
            doc_id (str): The ID of the document.
            chunk_size (int, optional): The maximum size of each chunk in characters. Defaults to 600.
            chunk_overlap (int, optional): The number of characters to overlap between chunks. Defaults to 150.
            min_chunk_size (int, optional): The minimum size of each chunk in characters. Defaults to 200.
            similarity_threshold (float, optional): The similarity threshold for determining chunk boundaries. Defaults to 0.5.
            debug (bool, optional): Whether to enable debug mode. Defaults to False.

        Paragraph upto chunk_size character is considered as a single chunk.
        If a paragraph short of chunk_size characters, it will be combined with the next paragraph to form a chunk of up to chunk_size characters, if the second paragraph can be added without breaking sentences.
        If a paragraph exceeds chunk_size characters, it will be split into multiple chunks, ensuring that each chunk ends at a sentence boundary and the succeding chunk starts with an overlap of chunk_overlap characters from preceeding chunk without breaking a sentence with a 10% character tolerance.
        if a chunk spans multiple pages, the page number will be recorded as the starting page.
        The chunking logic ensures that sentences are not broken and that overlaps are maintained for context continuity.
        Returns:
            list[Chunk]: A list of Chunk objects containing the chunked paragraphs, chunk id, section_tag, page number and doc_id.

    """
    flat = []
    for page_number, paragraphs in section:
        for paragraph in paragraphs:
            sentences = re.split(r"(?<!\b[A-Z]\.)(?<=[.!?])\s+(?=[A-Z])", paragraph)
            for sentence in sentences:
                if len(sentence) <= chunk_size:
                    flat.append((page_number, sentence))
                else:
                    words = sentence.split(" ")
                    current_chunk = ""
                    for word in words:
                        if len(current_chunk) + len(word) + 1 <= chunk_size:
                            current_chunk += word + " "
                        else:
                            if current_chunk:
                                flat.append((page_number, current_chunk.strip()))
                            current_chunk = word + " "
                    if current_chunk:
                        flat.append((page_number, current_chunk.strip()))
    buffer = []
    counter = 0
    chunks = []
    embedding_texts = []
    if not flat:
        return []
    for page_number, sentence in flat:
        embedding_text = "search_document: " + sentence
        embedding_texts.append(embedding_text)
    embed_sentences = model.encode(embedding_texts, 
                        normalize_embeddings=True, 
                        batch_size=16,
                        show_progress_bar=False)
    sims = np.sum(embed_sentences[1:] * embed_sentences[:-1], axis=1)
    if debug and len(sims) > 0:
        print("Similarity percentiles:", np.percentile(sims, [10, 20, 30, 50, 70, 90]))
        return []
    for idx, (page_number, sentence) in enumerate(flat):
        string = " ".join([s for _, s in buffer])
        if not buffer:
            size = len(string) + len(sentence)
        else:
            size = len(string) + len(sentence) + 1
            similarity = sims[idx-1]
        if(size <= chunk_size and (not buffer or similarity > similarity_threshold)):
            buffer.append((page_number, sentence))
        elif size > chunk_size and buffer and similarity > similarity_threshold:
            chunk_text = " ".join([s for _, s in buffer])
            chunk = Chunk(
                text=chunk_text,
                chunk_id=doc_id+f"_{section_tag}_{counter:04d}",
                section_tag=section_tag,
                page_number=buffer[0][0],
                doc_id=doc_id
            )
            pg_no, txt = buffer[-1]
            buffer = []
            if len(txt) <= chunk_overlap * 1.10 and len(sentence + txt) + 1 <= chunk_size:
                buffer.append((pg_no, txt))
            counter += 1
            buffer.append((page_number, sentence))
            chunks.append(chunk)
        elif size > chunk_size and buffer and similarity <= similarity_threshold:
            chunk_text = " ".join([s for _, s in buffer])
            chunk = Chunk(
                text=chunk_text,
                chunk_id=doc_id+f"_{section_tag}_{counter:04d}",
                section_tag=section_tag,
                page_number=buffer[0][0],
                doc_id=doc_id
            )
            buffer = []
            counter += 1
            buffer.append((page_number, sentence))
            chunks.append(chunk)
        elif size <= chunk_size and len(string) > min_chunk_size and similarity <= similarity_threshold:
            chunk_text = " ".join([s for _, s in buffer])
            chunk = Chunk(
                text=chunk_text,
                chunk_id=doc_id+f"_{section_tag}_{counter:04d}",
                section_tag=section_tag,
                page_number=buffer[0][0],
                doc_id=doc_id
            )
            buffer = []
            counter += 1
            buffer.append((page_number, sentence))
            chunks.append(chunk)
        elif size <= chunk_size and len(string) <= min_chunk_size and similarity <= similarity_threshold:
            buffer.append((page_number, sentence))
    if buffer:
        chunk = Chunk(
            text=" ".join([s for _, s in buffer]),
            chunk_id=doc_id+f"_{section_tag}_{counter:04d}",
            section_tag=section_tag,
            page_number=buffer[0][0],
            doc_id=doc_id
        )
        chunks.append(chunk)
    return chunks

        

In [29]:
def build_index(chunks: list[Chunk]) -> None:
    """Builds an index for the given list of chunks. Store vector embeddings for each chunk as numpy arrays and store the chunk text and metadatas as json file.
    The function expects the chunks from all 21 papers to be included in the list.

        Args:
            chunks (list[Chunk]): A list of Chunk objects to be indexed.
        Model used for embedding the chunk is nomic-ai/nomic-embed-text-v1.5
        The vector embeddings row should match the order of metadata entries.
        The numpy arrays and metadata json will be stored as separate files under rag-foundation/output.
        Returns:
            None

    """
    metadata_json = []
    embedding_texts = []
    for chunk in chunks:
        chunk_text = chunk.text
        embedding_text = "search_document: " + chunk_text
        embedding_texts.append(embedding_text)
        metadata_json.append({
            "chunk_id": chunk.chunk_id,
            "text": chunk.text,
            "doc_id": chunk.doc_id,
            "page_number": chunk.page_number,
            "section_tag": chunk.section_tag
        })
    embeddings = model.encode(embedding_texts, 
                        normalize_embeddings=True, 
                        batch_size=16,
                        show_progress_bar=True)
    print(f"Number of embedding texts: {len(embedding_texts)}")
    print(f"Shape of embeddings: {embeddings.shape}")
    assert len(embedding_texts) == embeddings.shape[0]
    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH, exist_ok=True)
    np.save(f"{OUTPUT_PATH}/index.npy", embeddings)
    with open(f"{OUTPUT_PATH}/metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata_json, f)

In [34]:
grand_total = 0
all_chunks = []
for idx, path in enumerate(paths):
    if idx < 5:
        print(f"Processing file {idx}: {path}")
        doc_id = path.stem
        pages = load_pages(path)
        sections = split_sections(pages)
        section_counts = {"Content": 0, "References": 0, "Appendix": 0}

        for section_name, section_pages in sections.items():
                print(section_name)
                cleaned_section = clean_section(section_pages)
                chunks = semantic_chunking(cleaned_section, section_name, doc_id, chunk_size=600, chunk_overlap=150, min_chunk_size=200, similarity_threshold=0.5, debug=True)

Processing file 0: /Users/tomthomas/Documents/rag-foundation/data/papers/2603.10913.pdf
Content
Similarity percentiles: [0.58279321 0.63603296 0.66985887 0.71606779 0.75185974 0.81698714]
References
Similarity percentiles: [0.48714025 0.50667101 0.52728727 0.59267783 0.64519188 0.79588349]
Appendix
Similarity percentiles: [0.57745237 0.61696651 0.65404359 0.7140168  0.76876855 0.83839564]
Processing file 1: /Users/tomthomas/Documents/rag-foundation/data/papers/2409.14924.pdf
Content
Similarity percentiles: [0.64788585 0.67595491 0.70244703 0.73537111 0.77216576 0.83038568]
References
Similarity percentiles: [0.50462742 0.52219512 0.536585   0.55581045 0.57902832 0.6388962 ]
Appendix
Processing file 2: /Users/tomthomas/Documents/rag-foundation/data/papers/2510.12323.pdf
Content
Similarity percentiles: [0.61301745 0.64354143 0.66107717 0.70059925 0.72910719 0.78435812]
References
Similarity percentiles: [0.47517231 0.49107257 0.50012305 0.52481496 0.54285777 0.57592566]
Appendix
Similari

In [10]:
grand_total = 0
all_chunks = []
for path in paths:
    doc_id = path.stem
    pages = load_pages(path)
    sections = split_sections(pages)
    section_counts = {"Content": 0, "References": 0, "Appendix": 0}

    for section_name, section_pages in sections.items():
        cleaned_section = clean_section(section_pages)
        chunks = chunk_paragraphs(cleaned_section, section_name, doc_id)
        section_counts[section_name] = len(chunks)
        all_chunks.extend(chunks)

    paper_total = sum(section_counts.values())
    grand_total += paper_total
    print(
        f"{doc_id}  content={section_counts['Content']}  "
        f"refs={section_counts['References']}  "
        f"appendix={section_counts['Appendix']}  total={paper_total}"
    )
print(len(all_chunks))
build_index(all_chunks)
print(f"Grand total: {grand_total}")

2603.10913  content=77  refs=32  appendix=93  total=202
2409.14924  content=161  refs=95  appendix=0  total=256
2510.12323  content=110  refs=8  appendix=38  total=156
2310.11511  content=98  refs=34  appendix=96  total=228
2601.17036  content=89  refs=15  appendix=39  total=143
2608.12875  content=77  refs=47  appendix=137  total=261
2505.03275  content=57  refs=7  appendix=0  total=64
2403.14403  content=100  refs=34  appendix=18  total=152
2601.12658  content=89  refs=19  appendix=0  total=108
2312.10997  content=164  refs=76  appendix=0  total=240
2401.15884  content=86  refs=26  appendix=22  total=134
2607.23121  content=92  refs=14  appendix=0  total=106
2608.27460  content=42  refs=10  appendix=0  total=52
2507.23386  content=79  refs=35  appendix=61  total=175
2604.05096  content=77  refs=22  appendix=44  total=143
2402.06196  content=344  refs=101  appendix=19  total=464
2407.16833  content=65  refs=21  appendix=20  total=106
2604.15484  content=98  refs=6  appendix=0  total=1

Batches: 100%|██████████| 230/230 [06:39<00:00,  1.74s/it]

Number of embedding texts: 3678
Shape of embeddings: (3678, 768)
Grand total: 3678


In [4]:
def load_index(path: str) -> tuple[np.ndarray, list[Chunk]]:
    """Loads the index and metadata from the given path.

        Args:
            path (str): The path to the index files - rag-foundation/output.

        The path corresponds to the directory where the index files are stored should be same as during the build_index function.
        The index files are expected to be in the format of numpy arrays for vector embeddings and JSON files for metadata.
        Returns:
            tuple[np.ndarray, list[Chunk]]: A tuple containing the vector embeddings as a numpy array and the metadata as a list of chunks.

    """
    index = np.load(f"{path}/index.npy")
    with open(f"{path}/metadata.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)
    chunks = [Chunk(**chunk_data) for chunk_data in metadata]
    assert len(index) == len(chunks)
    return index, chunks

In [5]:
def retrieve(query: str, index: np.ndarray, chunks: list[Chunk], top_k: int = 5, section_tag: str = "Content") -> list[RetrievedChunk]:
    """Retrieves the top_k most similar chunks to the query using the index.

        Args:
            query (str): The query string to search for.
            index (np.ndarray): The numpy array containing vector embeddings.
            chunks (list[Chunk]): The list of chunks corresponding to the index.
            top_k (int, optional): The number of top results to return. Defaults to 5.
            section_tag (str, optional): The section tag to consider for similarity. Defaults to "Content".
        Use the embedding model loaded nomic-ai/nomic-embed-text-v1.5 to embed the query.
        Vectors are normalized before computing similarity.
        User query need to be changed as search_query before passing to model
        Sort the chunks based on their similarity score to the query embedding.
        Returns:
            list[RetrievedChunk]: A list of RetrievedChunk objects representing the top_k most similar chunks.

    """
    query = f"search_query: {query}"
    query_embedding = model.encode([query], normalize_embeddings=True)[0]
    idx = [i for i, chunk in enumerate(chunks) if chunk.section_tag == section_tag]
    if not idx:
        raise ValueError(f"No chunks found with section_tag: {section_tag}")
    chunks_filtered = [chunks[i] for i in idx]
    index_filtered = index[idx]
    similarities = index_filtered @ query_embedding
    top_indices = similarities.argsort()[-top_k:][::-1]
    retrieved_chunks = []
    for pos in top_indices:
        chunk = chunks_filtered[pos]
        retrieved_chunks.append(RetrievedChunk(chunk=chunk, score=np.float32(similarities[pos])))
    return retrieved_chunks

In [9]:
def check_citation(answer: Answer, retrieved_chunks: list[RetrievedChunk]) -> list[tuple[bool, Citation]]:
    """Checks the citation in the given answer.

        Args:
            answer (Answer): The answer object containing the answer sentences and citations.
            retrieved_chunks (list[RetrievedChunk]): The list of retrieved chunks relevant to the answer.
        Validates the citations to ensure they correctly reference the source chunks used in generating the answer.
        If any citations are incorrect or missing, they should be flagged for correction.
        Citation should contain page number and doc_id as per the retrieved chunks.
        A valid citation means that the page number and doc_id correctly correspond to the source chunk used in generating the answer.
        Returns:
            list[tuple[bool, Citation]]: A list of tuples containing a boolean indicating the validity of the citation and the corresponding citation object.

    """
    results = []
    for sentence in answer.sentences:
        for citation in sentence.citations:
            is_valid = False
            for chunk in retrieved_chunks:
                if chunk.chunk.doc_id == citation.doc_id and chunk.chunk.page_number == citation.page_number:
                    is_valid = True
                    break
            results.append((is_valid, citation))
    return results

In [17]:
def answer(query: str, retrieved_chunks: list[RetrievedChunk]) -> Answer:
    """Generates an answer sentence based on the search query and retrieved chunks.

        Args:
            query (str): The search query string.
            retrieved_chunks (list[RetrievedChunk]): The list of retrieved chunks relevant to the search query.

        Ollama model 'llama3.2:3b' is used to generate the answer based on the retrieved chunks and the query.
        The answer is generated based on the content of the retrieved chunks and the query.
        Citations are included to reference the source chunks used in generating the answer.
        If the retrieved chunks do not contain sufficient information to answer the query, respond  saying Sorry context doesn't have enough details to answer the question.
        Validate the model output to ensure it returns the expected outcome as per the Answer model.
        Returns:
            Answer: A list of generated answer sentences and corresponding citations.

    """
    prompt = f"You are an AI assistant. Answer the following user query based on the provided context."
    rule = "\nRules:\n - Only use the information provided in the context to answer the query.\n - Include citations to the relevant chunks.\n - If the information is insufficient, respond with the refusal message. \n - Copy doc_id and page exactly as in the chunk headers. \n - Always return at least one sentence; a refusal is one sentence with an empty citations list. \n - Respond in JSON with a sentences list, where each item has text and a citations list of doc_id and page_number. \n - Answer only if the chunk/context directly answers the user query, never try to link an irrelevant content with user query."
    prompt += rule
    context_prompt = "\nContext:"
    prompt += context_prompt
    for i, chunk in enumerate(retrieved_chunks):
        prompt += f"\nChunk Text: ({chunk.chunk.doc_id},{chunk.chunk.page_number}) {chunk.chunk.text}"
    refusal_prompt = "\nIf the retrieved chunks do not contain sufficient information to answer the query, respond with 'Sorry context doesn't have enough details to answer the question.'"
    prompt += refusal_prompt
    output_prompt = "\nPlease provide the answer with citations to the relevant chunks, citations should include doc_id and page_number."
    prompt += output_prompt
    prompt += f"\nQuery: {query}"
    response = chat(
        model='llama3.2:3b',
        format=Answer.model_json_schema(),
        options={'temperature': 0.0},
        messages=[{'role': 'user', 'content': prompt}],
    )
    answer_text = response.message.content
    return Answer.model_validate_json(answer_text)


In [19]:
index, chunks = load_index(OUTPUT_PATH)
questions = ["When is diwali celebrated in india?",
             "What is HNSW?",
             "What is multimodal knowledge unification?",
             "How RAG MCP evaluated, what are the different metrices used?",
             "What is time aware retrieval?",
             "How RAG supports a doctor in diagnosis?"]
for question in questions:
    retrieved_chunks = retrieve(question, index, chunks, top_k=5, section_tag="Content")
    print(question)
    result = answer(question, retrieved_chunks)
    citation_check_results = check_citation(result, retrieved_chunks)
    print(citation_check_results)
    if any(not is_valid for is_valid, _ in citation_check_results):
        print("Citation check failed for this question.")
    if not citation_check_results:
        print("Refusal, no citations expected.")
    else:
        print("All citations were in the retrieved set.")

When is diwali celebrated in india?
[]
Refusal, no citations expected.
What is HNSW?
[(True, Citation(page_number=2, doc_id='2608.27460'))]
All citations were in the retrieved set.
What is multimodal knowledge unification?
[(True, Citation(page_number=2, doc_id='2510.12323'))]
All citations were in the retrieved set.
How RAG MCP evaluated, what are the different metrices used?
[(True, Citation(page_number=9, doc_id='2505.03275'))]
All citations were in the retrieved set.
What is time aware retrieval?
[(True, Citation(page_number=2, doc_id='2604.05096'))]
All citations were in the retrieved set.
How RAG supports a doctor in diagnosis?
[(True, Citation(page_number=5, doc_id='2312.10997')), (True, Citation(page_number=5, doc_id='2312.10997'))]
All citations were in the retrieved set.


In [64]:
answer

Answer(sentences=[])

In [34]:
for question in questions:
    retrieved_chunks = retrieve(question, index, chunks, top_k=5, section_tag="Content")
    print(f"Question: {question}")
    print("Retrieved Chunks:")
    for chunk in retrieved_chunks:
        chunk_text = chunk.chunk.text
        page_number = chunk.chunk.page_number
        print(f"Chunk Text: {chunk_text}")
        doc_id = chunk.chunk.doc_id
        print(f"Score: {chunk.score}, Doc ID: {doc_id}, Page Number: {page_number}, Chunk: {chunk_text[:200]}")


Question: When is diwali celebrated in india?
Retrieved Chunks:
Chunk Text: Avg Narr Qasp Mult Hotp 2Wiki Musi Sum En.QA En.MC 1-1 LC 49.70 32.76 47.83 52.33 61.85 62.96 40.22 20.73 43.08 85.57 Gemini-1.5-Pro 1-2 RAG 37.33 22.54 44.68 49.53 48.36 54.24 26.56 19.51 19.46 51.09 1-3 SELF-ROUTE 46.41 28.32 45.23 51.47 55.18 62.68 40.66 19.77 37.51 76.86 1-4 answerable % 76.78 73.00 85.00 96.67 84.50 81.00 58.50 93.50 56.41 62.45 1-5 token % 38.39 23.07 49.93 36.88 32.97 53.49 56.14 17.96 42.25 32.84 GPT-4O 2-1 LC 48.67 32.78 44.54 55.28 62.42 70.69 41.65 21.92 32.36 76.42 2-2 RAG 32.60 18.05 46.02 50.74 36.86 50.21 16.09 19.97 14.43 41.05 2-3 SELF-ROUTE 48.89 31.36 47.99
Score: 0.5631457567214966, Doc ID: 2407.16833, Page Number: 4, Chunk: Avg Narr Qasp Mult Hotp 2Wiki Musi Sum En.QA En.MC 1-1 LC 49.70 32.76 47.83 52.33 61.85 62.96 40.22 20.73 43.08 85.57 Gemini-1.5-Pro 1-2 RAG 37.33 22.54 44.68 49.53 48.36 54.24 26.56 19.51 19.46 51.09
Chunk Text: Contemporary Questions C1: Who was the wo

In [37]:
from ollama import chat

response = chat(
    model='llama3.2:3b',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)
print(response.message.content)

Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?
